# 🔍 Notebook 3 — Vector Store & Retrieval Verification

Test your full pipeline end-to-end:
- Confirm vector store has correct doc count
- Run vector, BM25, and hybrid searches
- Batch sanity check — 6 real support queries vs expected sources

**Requires:** `ingest.py` completed (vector_db/ and parents.pkl exist)

In [1]:
import os, pickle, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

def find_backend() -> Path:
    for p in [Path(os.getcwd()).resolve()] + list(Path(os.getcwd()).resolve().parents):
        if (p / 'parent_docstore').exists() and (p / 'data').exists():
            return p
        if (p / 'backend' / 'parent_docstore').exists():
            return p / 'backend'
    raise RuntimeError('Cannot find backend root.')

BACKEND     = find_backend()
DB_PATH     = BACKEND / 'vector_db'
PARENTS_PKL = BACKEND / 'parents.pkl'
PARENT_STORE_PATH = BACKEND / 'parent_docstore'

for p in [DB_PATH, PARENT_STORE_PATH, PARENTS_PKL]:
    status = '✅' if p.exists() else '❌ MISSING — run ingest.py first'
    print(f'{status}  {p.name}')

✅  vector_db
✅  parent_docstore
✅  parents.pkl


In [2]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.storage import LocalFileStore
from langchain.storage._lc_store import create_kv_docstore

print('Loading embedding model (first run downloads ~90 MB)…')
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
)
vectorstore = Chroma(persist_directory=str(DB_PATH), embedding_function=embeddings)
docstore    = create_kv_docstore(LocalFileStore(str(PARENT_STORE_PATH)))

with open(PARENTS_PKL, 'rb') as f:
    parents_list = pickle.load(f)

n_vectors = vectorstore._collection.count()
print(f'\n✅  Vectors       : {n_vectors}')
print(f'✅  Parent docs   : {len(parents_list)}')

if n_vectors != len(parents_list):
    print(f'⚠️  Mismatch! Expected {len(parents_list)} vectors. Re-run ingest.py.')
else:
    print('✅  Counts match — store is healthy')

Loading embedding model (first run downloads ~90 MB)…


C:\Users\karth\AppData\Local\Temp\ipykernel_25664\886824467.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
C:\Users\karth\AppData\Local\Temp\ipykernel_25664\886824467.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=str(DB_PATH), embedding_function=embeddings)
Failed to send telemet


✅  Vectors       : 85
✅  Parent docs   : 85
✅  Counts match — store is healthy


In [3]:
# ── Vector search ───────────────────────────────────────────────────────────
QUERY = 'how do I reset my thermostat to factory settings'
TOP_K = 4

print(f'Query: "{QUERY}"\n' + '─'*60)
for rank, (doc, score) in enumerate(vectorstore.similarity_search_with_score(QUERY, k=TOP_K), 1):
    section = doc.metadata.get('section_title') or doc.metadata.get('Category','?')
    print(f'[{rank}] score={score:.4f}  {doc.metadata.get("source","?")} › {section}')
    print(f'     {doc.page_content[:180].strip()}…\n')

Query: "how do I reset my thermostat to factory settings"
────────────────────────────────────────────────────────────


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[1] score=0.5426  faqs.csv › Thermostat
     To reset the Thermostat Pro, press and hold the reset button for 10 seconds until the screen blinks, then release, following the step-by-step procedure to restore default settings…

[2] score=1.1175  faqs.csv › Thermostat
     To update the firmware on the Thermostat Pro, go to the Nexora app, navigate to Settings, select Device, and choose Update Firmware, following these step-by-step procedures for the…

[3] score=1.1959  faqs.csv › Thermostat
     The Thermostat Pro supports temperature and humidity controls when connected to compatible HVAC systems, with specific model numbers and compatibility details available in the prod…

[4] score=1.2080  faqs.csv › Troubleshooting
     The Thermostat Pro screen may be blank due to improper power supply, so check if it is properly powered, and if not, ensure the wiring is correct by verifying the connections, or c…



In [4]:
# ── BM25 keyword search ─────────────────────────────────────────────────────
from langchain_community.retrievers import BM25Retriever

bm25  = BM25Retriever.from_documents(parents_list, k=4)
QUERY = 'warranty void water damage'

print(f'BM25 Query: "{QUERY}"\n' + '─'*60)
for rank, doc in enumerate(bm25.invoke(QUERY), 1):
    section = doc.metadata.get('section_title') or doc.metadata.get('Category','?')
    print(f'[{rank}] {doc.metadata.get("source","?")} › {section}')
    print(f'     {doc.page_content[:180].strip()}…\n')

BM25 Query: "warranty void water damage"
────────────────────────────────────────────────────────────
[1] visionsphere_360_manual.md › 8. Warranty & Support
     ## 8. Warranty & Support
-   **Warranty:** Nexora provides a **1-year limited manufacturer's warranty** from the date of purchase. This warranty covers hardware defects under norma…

[2] faqs.csv › Warranty
     Question: Are accidental damages covered under warranty?
Answer: No, warranty excludes accidental damage, water damage, and unauthorized repairs.…

[3] lumiglow_smart_lighting_manual.md › 8. Warranty & Support
     ## 8. Warranty & Support
-   **Warranty:** This product is covered by a 1-year limited manufacturer's warranty from the date of purchase. The warranty covers defects in materials a…

[4] visionsphere_360_manual.md › 3. Safety, Placement & Privacy
     ## 3. Safety, Placement & Privacy
**WARNING: Please read all guidelines before installation to ensure optimal performance and safety.**

### 3.1 Power & Placem

In [5]:
# ── Hybrid search (mirrors your actual rag.py pipeline) ─────────────────────
from langchain.retrievers import EnsembleRetriever

hybrid = EnsembleRetriever(
    retrievers=[
        vectorstore.as_retriever(search_kwargs={'k': 4}),
        BM25Retriever.from_documents(parents_list, k=4),
    ],
    weights=[0.6, 0.4],
)

QUERY = 'LumiGlow light not turning on after setup'
print(f'Hybrid Query: "{QUERY}"\n' + '─'*60)
for rank, doc in enumerate(hybrid.invoke(QUERY), 1):
    section = doc.metadata.get('section_title') or doc.metadata.get('Category','?')
    print(f'[{rank}] {doc.metadata.get("source","?")} › {section}')
    print(f'     {doc.page_content[:180].strip()}…\n')

Hybrid Query: "LumiGlow light not turning on after setup"
────────────────────────────────────────────────────────────
[1] lumiglow_smart_lighting_manual.md › 6. In-Depth Troubleshooting
     The Lumiglow smart lighting system may experience connectivity issues, such as the bulb not entering pairing mode, which can be resolved by performing a manual reset, confirming th…

[2] lumiglow_smart_lighting_manual.md › 1. Introduction
     The LumiGlow Smart Light is part of the Nexora smart home ecosystem, engineered to provide vibrant, responsive, and energy-efficient illumination, with this manual guiding you thro…

[3] lumiglow_smart_lighting_manual.md › 3. Safety & Compliance
     The Lumiglow smart lighting system requires careful installation and use, with warnings against electric shock, water exposure, and overheating, and must be operated in fixtures ra…

[4] lumiglow_smart_lighting_manual.md › 4. Installation & Initial Setup
     For the LumiGlow Smart Light, ensure the power is off

In [6]:
# ── Batch sanity check ───────────────────────────────────────────────────────
# Verifies the right documents surface for 6 realistic support queries.
# Screenshot this for your portfolio!
test_queries = [
    # (query, expected_source_substring)
    # faqs.csv counts as valid for any topic — it's purpose-built for user queries
    ('thermostat factory reset steps',          ['nexora_thermostat', 'faqs']),
    ('LumiGlow light flickering fix',           ['lumiglow', 'faqs']),
    ('VisionSphere motion detection setup',     ['visionsphere', 'faqs']),
    ('return policy how many days',             ['policies', 'faqs']),
    ('warranty void water damage',              ['policies', 'faqs']),
    ('app not connecting to device wifi',       None),
]

print(f'{"Query":<46} {"Top-1 Source":<38} Pass?')
print('─'*95)

passed = 0
for query, expected in test_queries:
    top = vectorstore.similarity_search(query, k=1)
    if not top:
        print(f'{query:<46} {"NO RESULTS":<38} ❌')
        continue
    src = top[0].metadata.get('source', '?')
    ok = expected is None or any(e.lower() in src.lower() for e in expected)
    if ok: passed += 1
    verdict = '✅' if ok else f'⚠️  (got {src})'
    print(f'{query:<46} {src:<38} {verdict}')

print(f'\nPassed {passed}/{len(test_queries)} sanity checks')

Query                                          Top-1 Source                           Pass?
───────────────────────────────────────────────────────────────────────────────────────────────
thermostat factory reset steps                 faqs.csv                               ✅
LumiGlow light flickering fix                  lumiglow_smart_lighting_manual.md      ✅
VisionSphere motion detection setup            visionsphere_360_manual.md             ✅
return policy how many days                    faqs.csv                               ✅
warranty void water damage                     faqs.csv                               ✅
app not connecting to device wifi              faqs.csv                               ✅

Passed 6/6 sanity checks
